# Day 3: Prompt Engineering Mastery
Today we will learn advanced prompting techniques to improve model accuracy and build a useful utility.

## What You'll Learn:
1. **Zero-Shot vs Few-Shot Prompting:** Giving models examples of the output we expect.
2. **Chain of Thought (CoT):** Forcing the model to show its step-by-step thinking for logical problems.
3. **AI Code Reviewer:** Building a helper prompt to evaluate and improve Python code snippets.

In [2]:
import ollama

print("Day 3 workspace ready!")

Day 3 workspace ready!


## Zero-Shot vs Few-Shot Prompting
How do we teach an LLM to output exactly what we want?

1. **Zero-Shot Prompting:** We ask the model to perform a task *without* giving it any examples. We rely entirely on its general knowledge.
2. **Few-Shot Prompting:** We provide the model with a few examples of inputs and desired outputs first. This "in-context learning" helps the model copy the exact style, formatting, and tone we want.

Let's see this in action by building a "comment-to-emoji" classifier. We want the model to read a sentence and return ONLY a single emoji.

In [3]:
# The text we want to classify
test_comment = "I waited two hours for my food and it was cold!"

# 1. Zero-Shot Attempt (No Examples)
print("Zero-Shot Response:")
print("-" * 50)

zero_shot_prompt = f"""
Classify the sentiment of this text using a single emoji.
Text: "{test_comment}"
"""

response_zero = ollama.chat(
    model='llama3.2:1b',
    messages=[{'role': 'user', 'content': zero_shot_prompt}]
)
print(response_zero.message.content.strip())


print("\n" + "="*60 + "\n")


# 2. Few-Shot Attempt (Providing 3 Examples first)
print("Few-Shot Response:")
print("-" * 50)

few_shot_prompt = f"""
Classify the sentiment of the text using a single emoji. Here are some examples:

Text: "I absolutely love this keyboard, it works great!"
Emoji: 😍

Text: "It's okay, nothing special but works fine."
Emoji: 😐

Text: "This app crashes constantly, do not buy!"
Emoji: 😡

Now classify this one:
Text: "{test_comment}"
Emoji:"""

response_few = ollama.chat(
    model='llama3.2:1b',
    messages=[{'role': 'user', 'content': few_shot_prompt}]
)
print(response_few.message.content.strip())

Zero-Shot Response:
--------------------------------------------------
☹️


Few-Shot Response:
--------------------------------------------------
The sentiment of the text is neutral to negative. There's no explicit expression of love or praise, but there's also no explicit criticism or warning about a flaw in the app. The tone appears to be more disappointed and frustrated due to the poor timing and temperature of the food than it does with the app itself.


## Chain of Thought (CoT)
When we ask LLMs to solve logical puzzles, math problems, or complex reasoning tasks, they often make mistakes if they try to answer immediately. 

**Chain of Thought** prompting solves this by forcing the model to write down its reasoning steps *before* giving the final answer. 

By thinking out loud, the model uses more tokens for reasoning, which dramatically increases its accuracy. The simplest way to trigger this is by adding the phrase: **"Think step-by-step."**

In [4]:
# A simple word problem
math_problem = "A farmer has 15 chickens and 10 cows. How many legs are there in total on the farm?"

# Case A: Standard Prompt (Direct Answer)
print("Case A: Direct Prompt")
print("-" * 50)

prompt_direct = f"""
Solve this problem and output ONLY the final number:
"{math_problem}"
"""

response_direct = ollama.chat(
    model='llama3.2:1b',
    messages=[{'role': 'user', 'content': prompt_direct}]
)
print(response_direct.message.content.strip())


print("\n" + "="*60 + "\n")


# Case B: Chain of Thought Prompt (Step-by-Step)
print("Case B: Chain of Thought Prompt")
print("-" * 50)

prompt_cot = f"""
Solve this problem. Think step-by-step before writing the final answer.
"{math_problem}"
"""

response_cot = ollama.chat(
    model='llama3.2:1b',
    messages=[{'role': 'user', 'content': prompt_cot}]
)
print(response_cot.message.content.strip())

Case A: Direct Prompt
--------------------------------------------------
To find the total number of legs, we need to add the number of chicken legs and cow legs.

Chickens have 2 legs each, so 15 * 2 = 30
Cows have 4 legs each, so 10 * 4 = 40

Total legs: 30 + 40 = 70


Case B: Chain of Thought Prompt
--------------------------------------------------
To solve this problem, let's break it down into steps:

1. Calculate the number of legs for each animal.

- Chickens have 2 legs each.
- Cows have 4 legs each.

Step-by-step calculations:
- For chickens: 15 chickens * 2 legs/chicken = 30 legs
- For cows: 10 cows * 4 legs/cow = 40 legs

2. Sum the total number of legs from both animals to find the overall count.

Adding the calculated legs together:
- Total legs = 30 (chickens) + 40 (cows) = 70

Therefore, there are a total of 70 legs on the farm.


## Building an AI Code Reviewer
We will define a system prompt that turns the LLM into a Senior Python Developer. 

We will feed it a poorly written Python function (using slow loops and bad variable names) and ask it to review it.

In [4]:
# 1. Define the role and constraints in the system prompt
system_instructions = """
You are an expert Python code reviewer. For any code snippet provided:
1. Rate the code quality on a scale of 1 to 10.
2. List the top 2 issues with the code.
3. Show an improved, clean Python version of the code.
Keep your response concise and easy to read.
"""

# 2. A poorly written Python function (calculates average of a list using a slow loop)
bad_python_code = """
def calc(x):
    total = 0
    for i in range(len(x)):
        total = total + x[i]
    ans = total / len(x)
    return ans
"""

print("🔍 Reviewing code...")

# 3. Call Ollama with the system instructions and the bad code snippet
response = ollama.chat(
    model='llama3.2:1b',
    messages=[
        {'role': 'system', 'content': system_instructions},
        {'role': 'user', 'content': f"Please review this code:\n\n```python\n{bad_python_code}\n```"}
    ]
)

# 4. Print the review
print(response.message.content)

🔍 Reviewing code...
**Code Quality Rating: 6/10**

The provided code is a simple mathematical function to calculate the average of an array. However, here are two significant issues:

1. **Magic Number**: The value `len(x)` is used directly in the calculation without any explanation or justification. It would be better to define this as a constant or an attribute for clarity.
2. **Variable Naming**: The variable name `x` could be more descriptive than `i`. Consider renaming it to something like `numbers`.

**Improved Code:**
```python
def calculate_average(numbers):
    """
    Calculate the average of a list of numbers.

    Args:
        numbers (list): A list of numbers.

    Returns:
        float: The average of the input numbers.
    """
    if not isinstance(numbers, list):
        raise ValueError("Input must be a list")
    
    # Check for empty list
    if len(numbers) == 0:
        return 0
    
    total = sum(numbers)
    average = total / len(numbers)
    return average
